# (Neurosymbolic AI) Lab: LLMs + Datalog (clingo)

In this lab you'll build a system that combines:

- **An LLM** that reads English and translates it into a **Datalog** program.
- **clingo**, a logic engine that runs the program and works out exactly which facts follow.

```
English problem ──► LLM ──► Datalog program ──► clingo ──► True / False
                    ▲                  │
                    └── error messages ┘   (self-repair loop, Part 5)
```

Then you'll measure whether this hybrid beats asking the LLM directly, and look at where the errors go.

| Part | Topic |
|---|---|
| 0 | Setup |
| 1 | Datalog in clingo; the problem set; the Poole & Mackworth example |
| 2 | The LLM on its own: the ten problems, Poole & Mackworth, the deeper problems |
| 3 | The LLM as translator, and comparing translations |
| 4 | The hybrid system: LLM translates, clingo reasons |
| 5 | Self-repair loop |
| 6 | Evaluation: LLM-only vs. hybrid |
| 7 | Datalog vs. first-order logic: what changes? |



### Background: Logic-LM

**Logic-LM** (Pan et al., 2023) is the system this lab is modeled on. Its idea is to divide the work: the LLM *translates* a problem into a formal language, and a symbolic solver does the *reasoning*. It has four stages:

1. **Problem Formulator.** The LLM, prompted with a few examples, turns the natural-language problem into a symbolic program.
2. **Symbolic Reasoner.** A solver runs the program. Different problem types get different solvers: a logic-programming engine (Pyke), a first-order theorem prover (Prover9), a constraint solver, or the Z3 SMT solver.
3. **Result Interpreter.** The solver's output is mapped back to an answer choice.
4. **Self-Refiner.** If the solver reports an error, the error message goes back to the LLM to fix the program, for a few rounds. If the program still won't run, the system falls back to the LLM's own answer.

Across five reasoning benchmarks (ProofWriter, PrOntoQA, FOLIO, LogicalDeduction, AR-LSAT), Logic-LM reported large gains over using the LLM alone: an average of 39.2% over standard prompting and 18.4% over chain-of-thought prompting. The gains were biggest on problems with long reasoning chains. Its answers are also *faithful*: each one comes with a program that can be checked.

**Limits to keep in mind:** the solver is only as good as the translation. The Self-Refiner fixes programs that fail to run (syntax errors), not programs that run but say the wrong thing.

**This lab's version:** Datalog in clingo plays the role of the symbolic language and solver. `solve_with_repair` is a simplified Self-Refiner.

**Reference:** Liangming Pan, Alon Albalak, Xinyi Wang, and William Yang Wang. 2023. "Logic-LM: Empowering Large Language Models with Symbolic Solvers for Faithful Logical Reasoning." *Findings of the Association for Computational Linguistics: EMNLP 2023*, pp. 3806–3824. [arXiv:2305.12295](https://arxiv.org/abs/2305.12295) · Code: [github.com/teacherpeterpan/Logic-LLM](https://github.com/teacherpeterpan/Logic-LLM)

### Background: benchmark datasets for logical reasoning

**ProofWriter** (Tafjord, Dalvi Mishra and Clark, 2021)
Short, synthetic "theories" written in simple English (facts like *"Erin is young."* and rules like *"If someone is big and young then they are red."*), each with questions that need **0 to 5 steps** of reasoning. The answer comes with a proof. It has two versions:

- **Closed-world (CWA):** anything that can't be derived is false. This matches Datalog and clingo, so these problems translate directly into this lab's pipeline. The deeper problems in Part 2.4 are in this style.
- **Open-world (OWA):** answers can be True, False, or **Unknown**, as in first-order logic.

Because the language is templated, ProofWriter tests *reasoning depth* more than language understanding.

**FOLIO** (Han et al., 2022; EMNLP 2024)
About 1,400 problems **written by experts** in natural, varied English, each annotated with its **first-order logic** translation. Answers are True, False, or Unknown. The sentences are much harder to translate than ProofWriter's (quantifiers, "either…or", "no one who…"), so FOLIO tests *translation* as much as reasoning. Logic-LM saw the most syntax errors on FOLIO, which is where its Self-Refiner helped most. FOLIO needs a first-order prover such as Prover9 or Z3, not Datalog.

**ProntoQA** (Saparov and He, 2023)
Synthetic chains of "Every wumpus is a yumpus" rules with **made-up category names**, so the model can't use world knowledge. Chain length is controlled, which makes it easy to see where reasoning breaks down. The deeper problems in Part 2.4 are in this style.

| Dataset | Language | Logic | Answers | Tests mainly |
|---|---|---|---|---|
| ProofWriter | Templated | Rules with negation (CWA or OWA) | True / False (/ Unknown) | Reasoning depth |
| FOLIO | Natural, expert-written | First-order logic | True / False / Unknown | Translation + reasoning |
| ProntoQA | Templated, made-up words | Category chains | True / False | Reasoning depth, no world knowledge |

**References**

- Oyvind Tafjord, Bhavana Dalvi Mishra, and Peter Clark. 2021. "ProofWriter: Generating Implications, Proofs, and Abductive Statements over Natural Language." *Findings of ACL-IJCNLP 2021*. [arXiv:2012.13048](https://arxiv.org/abs/2012.13048)
- Simeng Han, Hailey Schoelkopf, Yilun Zhao, et al. 2024. "FOLIO: Natural Language Reasoning with First-Order Logic." *Proceedings of EMNLP 2024*. [arXiv:2209.00840](https://arxiv.org/abs/2209.00840) · Data: [huggingface.co/datasets/yale-nlp/FOLIO](https://huggingface.co/datasets/yale-nlp/FOLIO)
- Abulhair Saparov and He He. 2023. "Language Models Are Greedy Reasoners: A Systematic Formal Analysis of Chain-of-Thought." *ICLR 2023*. [arXiv:2210.01240](https://arxiv.org/abs/2210.01240)

## Part 0: Setup

In [ ]:
!nvidia-smi

### Note: We need to install clingo, also pandas is used to display results

In [ ]:
%pip install -q clingo pandas requests


### Not necessary to restart

In [ ]:
import os, re
import pandas as pd
import clingo
print("clingo", clingo.__version__)

In [ ]:
import llama_cpp; print(llama_cpp.__version__)    # latest is  0.3.36

### The language model

This lab uses **Gemma 4 E4B** from Google, in the instruction-tuned **QAT Q4_0 GGUF** version (`google/gemma-4-E4B-it-qat-q4_0-gguf` on Hugging Face). It runs **locally on the JupyterHub GPU** through `llama-cpp-python`, so no API key or internet access is needed.

| | |
|---|---|
| **Model** | Gemma 4 E4B, instruction-tuned ("it") |
| **Size** | 7.5B parameters in total, about 4B "effective". Large per-layer lookup tables stay in CPU memory, so only part of the model has to fit on the GPU. |
| **QAT, Q4_0** | Quantization-aware training: the model was trained to keep its accuracy when its weights are stored in 4 bits. The file is about 5 GB. |
| **GGUF** | The file format used by llama.cpp for running models, not for training them. |


`call_llm(prompt)` is the only function in the notebook that talks to the model. It uses `temperature=0`, so the same prompt gives the same answer every time.

**Note:** the model takes a minute or two to load. Load it **once**, and if you rerun the cell, restart the kernel first so you don't end up with two copies in memory.

In [ ]:
import os, re, glob
from llama_cpp import Llama

# Folder on the shared drive where the Gemma model was downloaded.


read_path = "/home/jovyan/shared/models/gguf/google--gemma-4-E4B-it-qat-q4_0-gguf"
#Since the model is in directory, rather than single file need.
gguf_files = [f for f in glob.glob(f"{read_path}/**/*.gguf", recursive=True)
              if "mmproj" not in os.path.basename(f)]
assert gguf_files, f"No .gguf model file found under {read_path}"
GGUF_PATH = gguf_files[0]
def available_cpus():
    # CPUs this session may really use (JupyterHub limits are often far below os.cpu_count())
    try:
        quota, period = open("/sys/fs/cgroup/cpu.max").read().split()
        if quota != "max":
            return max(1, int(quota) // int(period))
    except (OSError, ValueError):
        pass
    return len(os.sched_getaffinity(0))

print("Using", available_cpus(), "CPU threads")
print("Loading", GGUF_PATH, "...")
llm = Llama(model_path=GGUF_PATH,n_ctx=2048,  n_gpu_layers=-1, n_threads=available_cpus(), use_mmap=False, flash_attn =True, verbose=False)
#llm = Llama(model_path=GGUF_PATH, n_ctx=4096, n_threads=os.cpu_count(), verbose=False)
print(f"GGUF loaded")

DEMO_MODE = False   # other cells check this


import time
def call_llm(prompt, demo_key=None):
    t = time.time()
    out = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        temperature=0, max_tokens=400)
    u = out["usage"]
    print(f"   [LLM call: {u['prompt_tokens']} in, {u['completion_tokens']} out, {time.time()-t:.1f}s]")
    text = out["choices"][0]["message"]["content"]
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()


## Part 1: Datalog in clingo

Datalog uses Prolog-style syntax:

| Datalog | Meaning |
|---|---|
| `parent(ann, bob).` | a **fact** |
| `ancestor(X, Y) :- parent(X, Y).` | a **rule**: `:-` means "if", commas mean "and" |
| `X`, `Y` (uppercase) | variables |
| `ann`, `bob` (lowercase) | constants |
| `not penguin(X)` | negation: "penguin(X) can't be derived" |
| `X != Y` | X and Y are different |

`run_datalog` runs a program and returns every derived fact, or clingo's error messages.

##### The first time clingo is run it takes a little longer . Code below prints out all of the ancestor relations both explicit and implicit.

In [ ]:
def run_datalog(program):
    # Return (set of derived facts as strings, None) or (None, error/warning text).
    msgs = []
    ctl = clingo.Control(["--warn=all"], logger=lambda code, msg: msgs.append(msg.strip()))
    try:
        ctl.add("base", [], program)
        ctl.ground([("base", [])])
    except RuntimeError:
        return None, "\n".join(msgs) or "clingo could not parse the program."
    facts = set()
    ctl.solve(on_model=lambda m: facts.update(str(s) for s in m.symbols(atoms=True)))
    return facts, ("\n".join(msgs) if msgs else None)

family = '''
parent(ann, bob).  parent(bob, cat).  parent(cat, dan).
ancestor(X, Y) :- parent(X, Y).
ancestor(X, Z) :- ancestor(X, Y), parent(Y, Z).
'''
facts, warn = run_datalog(family)
print(sorted(f for f in facts if f.startswith("ancestor")))

#### PROBLEMS is a set of 10 relatively shallow logical reasoning problems
They each have an id, the English text, and the correct formalization in Datalog, and the label of the correct Boolean result.

In [ ]:
PROBLEMS = [
 {"id": "P1", "text": "Birds fly unless they are penguins. Every penguin is a bird. Pingu is a penguin. Tweety is a bird.",
  "question": "Can Tweety fly?",
  "gold": "bird(X) :- penguin(X).\nflies(X) :- bird(X), not penguin(X).\npenguin(pingu).\nbird(tweety).",
  "gold_query": "flies(tweety)", "label": "True"},
 {"id": "P2", "text": "Birds fly unless they are penguins. Every penguin is a bird. Pingu is a penguin. Tweety is a bird.",
  "question": "Can Pingu fly?",
  "gold": "bird(X) :- penguin(X).\nflies(X) :- bird(X), not penguin(X).\npenguin(pingu).\nbird(tweety).",
  "gold_query": "flies(pingu)", "label": "False"},
 {"id": "P3", "text": "Every student in the AI class has taken logic. Anyone who has taken logic knows quantifiers. Maya is a student in the AI class.",
  "question": "Does Maya know quantifiers?",
  "gold": "taken_logic(X) :- student(X), in_ai_class(X).\nknows_quantifiers(X) :- taken_logic(X).\nstudent(maya).\nin_ai_class(maya).",
  "gold_query": "knows_quantifiers(maya)", "label": "True"},
 {"id": "P4", "text": "Ann is Bob's parent. Bob is Cat's parent. Cat is Dan's parent. A parent is an ancestor, and an ancestor of someone's parent is also their ancestor.",
  "question": "Is Ann an ancestor of Dan?",
  "gold": "parent(ann,bob).\nparent(bob,cat).\nparent(cat,dan).\nancestor(X,Y) :- parent(X,Y).\nancestor(X,Z) :- ancestor(X,Y), parent(Y,Z).",
  "gold_query": "ancestor(ann,dan)", "label": "True"},
 {"id": "P5", "text": "Ann is Bob's parent. Bob is Cat's parent. Cat is Dan's parent. A parent is an ancestor, and an ancestor of someone's parent is also their ancestor.",
  "question": "Is Dan an ancestor of Ann?",
  "gold": "parent(ann,bob).\nparent(bob,cat).\nparent(cat,dan).\nancestor(X,Y) :- parent(X,Y).\nancestor(X,Z) :- ancestor(X,Y), parent(Y,Z).",
  "gold_query": "ancestor(dan,ann)", "label": "False"},
 {"id": "P6", "text": "Trains run one way only: maple to oak, oak to pine, pine to cedar. A rider can reach the next stop, and can reach any stop reachable from the next stop.",
  "question": "Can a rider at cedar reach maple?",
  "gold": "next(maple,oak).\nnext(oak,pine).\nnext(pine,cedar).\nreaches(X,Y) :- next(X,Y).\nreaches(X,Z) :- next(X,Y), reaches(Y,Z).",
  "gold_query": "reaches(cedar,maple)", "label": "False"},
 {"id": "P7", "text": "CS201 requires CS101. CS301 requires CS201 and Math101. A student can take a course only if they have passed every course it requires. Sam passed CS101 and CS201. Sam is a student.",
  "question": "Can Sam take CS301?",
  "gold": "requires(cs201,cs101).\nrequires(cs301,cs201).\nrequires(cs301,math101).\npassed(sam,cs101).\npassed(sam,cs201).\nstudent(sam).\ncourse(C) :- requires(C,_).\nmissing(S,C) :- student(S), requires(C,P), not passed(S,P).\ncan_take(S,C) :- student(S), course(C), not missing(S,C).",
  "gold_query": "can_take(sam,cs301)", "label": "False"},
 {"id": "P8", "text": "All cats are mammals. Felix is a cat. Rex is a dog.",
  "question": "Is Rex a mammal?",
  "gold": "mammal(X) :- cat(X).\ncat(felix).\ndog(rex).",
  "gold_query": "mammal(rex)", "label": "False"},
 {"id": "P9", "text": "A student is on probation if they failed a course and are not enrolled in tutoring. Jo failed Calculus. Jo is enrolled in tutoring.",
  "question": "Is Jo on probation?",
  "gold": "probation(S) :- failed(S,_), not tutoring(S).\nfailed(jo,calculus).\ntutoring(jo).",
  "gold_query": "probation(jo)", "label": "False"},
 {"id": "P10", "text": "Ann is the parent of Bob and of Cal. Two different people with the same parent are siblings.",
  "question": "Is Bob a sibling of Cal?",
  "gold": "parent(ann,bob).\nparent(ann,cal).\nsibling(X,Y) :- parent(P,X), parent(P,Y), X != Y.",
  "gold_query": "sibling(bob,cal)", "label": "True"},
]



#### Code below gives a query to datalog program

In [ ]:
def answer(program, query):
    # Add the query as a rule, so clingo warns if its predicate is never defined.
    facts, msg = run_datalog(program + f"\nquery_holds :- {query}.")
    if facts is None:
        return "Error", msg
    return ("True" if "query_holds" in facts else "False"), msg

for p in PROBLEMS:
    got, _ = answer(p["gold"], p["gold_query"])
    print(p["id"], p["label"].ljust(5), "clingo:", got, "" if got == p["label"] else "  <-- MISMATCH")

#### The code below makes the problems easier to read and understand

In [ ]:
import pandas as pd
from IPython.display import display

def show_problems(problems):
    """Show id, text, question and answer for a list of problems as a table."""
    df = pd.DataFrame(problems)[["id", "text", "question", "label"]]
    df.columns = ["ID", "Text", "Question", "Answer"]
    display(df.style.hide(axis="index")
              .set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
              .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

def show_problem(pid, problems=None):
    """Print one problem in full, with its gold Datalog program."""
    problems = problems or PROBLEMS + globals().get("DEEP_PROBLEMS", [])
    p = next(q for q in problems if q["id"] == pid)
    print(f"{p['id']}: {p['text']}\nQuestion: {p['question']}\nAnswer:   {p['label']}\n")
    print("Gold program:")
    print("\n".join("    " + line for line in p["gold"].splitlines()))
    print(f"\nQuery:    {p['gold_query']}")

show_problems(PROBLEMS)

# show_problem("P7")              # one problem with its gold program

#### If you want to see the formalization in datalog, need to run show_problem on the individual one.

In [ ]:
show_problem('P5')

### The closed-world assumption

Datalog assumes that **anything it can't derive is false**. That makes `not` easy and gives every yes/no question an answer, but it's a different stance from first-order logic, where an underivable statement is just *unknown*. Note that in the problem set above all of the False conclusions depend on the closed world assumption.

In [ ]:
birds = '''
bird(X) :- penguin(X).
flies(X) :- bird(X), not penguin(X).
penguin(pingu).  bird(tweety).
'''
facts, _ = run_datalog(birds)
for q in ["flies(tweety)", "flies(pingu)", "flies(rex)"]:
    print(q, q in facts)
# In first-order logic, "Tweety can fly" was Unknown (Tweety might be a penguin).
# Under the closed-world assumption, Tweety isn't known to be a penguin, so it flies.

**Try it.** Add `penguin(tweety).` and rerun. Then try a typo such as a missing period, to see what clingo's error messages look like.

### Example from Poole & Mackworth: where is Kim?

This example is from Poole and Mackworth's textbook. In words:

- Kim is in room r123.
- Room r123 is part of the CS building.
- **Rule:** X is in Y if X is in some Z, and Z is part of Y.

**Query:** Is Kim in the CS building? The answer should be **True**.

The book writes the rule as `in(X,Y) ← part_of(Z,Y) ∧ in(X,Z)`. To write it in clingo:

| Book | clingo |
|---|---|
| `←` ("if") | `:-` |
| `∧` ("and") | `,` (comma) |
| every statement | ends with a period |

Below is the translation into Datalog

in(kim, r123).

part_of(r123, cs_building).

in(X, Y) :- part_of(Z, Y), in(X, Z).

Here it is as input to Clingo

In [ ]:

kim_book = """
in(kim, r123).
part_of(r123, cs_building).

in(X, Y) :- part_of(Z, Y), in(X, Z).
"""

result, msg = answer(kim_book, "in(kim,cs_building)")
print("in(kim,cs_building) is", result)          # True

#### Here is the rule with an error in translation.

In [ ]:
# Same rule, but part_of(X, Y) instead of part_of(Z, Y).
# Now it says: X is in Y if X itself is part of Y, and X is in something.
kim_typo = """
in(kim, r123).
part_of(r123, cs_building).

in(X, Y) :- part_of(X, Y), in(X, Z).
"""

result, msg = answer(kim_typo, "in(kim,cs_building)")
print("in(kim,cs_building) is", result)          # False
print("clingo messages:", msg)                   # None: no error, no warning

**What happened?** The second version would need *Kim herself* to be part of the CS building. No fact says that, so under the closed-world assumption the query is **False**.

Notice that clingo gave **no error and no warning**. The program is valid Datalog; it just means something different. This is the same kind of mistake an LLM translator can make, and the self-repair loop can't catch it, because there is no error message to send back.

**Try it.**

1. Add `part_of(cs_building, campus).` to the book's version and ask `in(kim,campus)`. How many times does the rule have to fire?
2. Add `in(lee, cs_building).` to the book's version. Is Lee in r123? Why not?

## Part 2: The LLM on its own

Before adding a solver, let's see how the LLM handles the problems by itself, as the **reasoner**. We go from easy to hard:

1. **The ten basic problems** (1–2 reasoning steps), compared with the gold answers.
2. **The Poole & Mackworth example** (one step), given in English and as a Datalog program.
3. **A deeper Poole & Mackworth example** (up to 6 steps).
4. **Seven deeper problems** (5–7 steps).

We ask in two ways:

- **Direct:** answer True or False and nothing else.
- **Step by step** (chain-of-thought prompting): explain the reasoning first, then give the answer.

Watch for: Does it follow the closed-world assumption? Does it bring in knowledge that isn't in the text? Is every step correct, even when the final answer is right? Unlike a solver's answer, there's no program here to check.

### 2.1 The ten basic problems

`ask_direct` gives the LLM a problem's text and question and asks for a one-word answer. We run it on all ten problems and compare with the gold labels.

In [ ]:
DIRECT_PROMPT = """Answer the question using only the text. Treat anything not stated as false.

Text: {text}
Question: {question}

Answer with exactly one word: True or False."""

def ask_direct(problem):
    """Ask the LLM to answer on its own, with no solver."""
    reply = call_llm(DIRECT_PROMPT.format(**problem))
    m = re.search(r"\b(True|False)\b", reply, re.I)
    return m.group(1).capitalize() if m else "Unparsed"

llm_basic = pd.DataFrame([{"ID": p["id"], "Question": p["question"],
                           "Answer key": p["label"], "LLM's answer": ask_direct(p)}
                          for p in PROBLEMS])
llm_basic["LLM right?"] = llm_basic["LLM's answer"] == llm_basic["Answer key"]
display(llm_basic)
print(f"LLM-only accuracy on the ten problems: {llm_basic['LLM right?'].mean():.0%}")

#### A closer look at one problem

Pick a problem the LLM got wrong above. First ask for a direct answer, then ask it to reason step by step. Does the step-by-step answer change? Is every step correct?

In [ ]:
PID = "P1"        # try others too, e.g. "P7" or "P8"

p = next(q for q in PROBLEMS + globals().get("DEEP_PROBLEMS", []) if q["id"] == PID) #Deep problems will be introduced later
print(f"Text:     {p['text']}\nQuestion: {p['question']}\nGold:     {p['label']}\n")

direct = call_llm(f"""Answer the question using only the text. Treat anything not stated as false.

Text: {p['text']}
Question: {p['question']}

Answer with exactly one word: True or False.""")
print("LLM, direct answer:", direct)

In [ ]:
t = time.time()
out = llm.create_chat_completion(
    messages=[{"role": "user", "content": f"""Answer the question using only the text. Treat anything not stated as false.

Text: {p['text']}
Question: {p['question']}

Reason step by step, one short numbered step per line. Then give the final answer on its own line as:
ANSWER: True   or   ANSWER: False"""}],
    temperature=0, max_tokens=500)
reasoning = out["choices"][0]["message"]["content"].strip()
print(reasoning)

m = re.search(r"ANSWER:\s*(True|False)", reasoning, re.I)
final = m.group(1).capitalize() if m else "Unparsed"
print(f"\nStep-by-step answer: {final}   Gold: {p['label']}   "
      f"({out['usage']['completion_tokens']} tokens, {time.time()-t:.0f} s)")

# For comparison: the solver on the gold program
print("clingo on the gold program:", answer(p["gold"], p["gold_query"])[0])

### 2.2 Poole & Mackworth: where is Kim?

The same one-step example from Part 1, given to the LLM in two forms: in English, and as the Datalog program itself.

In [ ]:
# The LLM on its own, on the Poole & Mackworth example (gold answer: True).

english_prompt = """Answer the question using only the text. Treat anything not stated as false.

Text: Kim is in room r123. Room r123 is part of the CS building.
Anyone who is in a place is also in anything that place is part of.
Question: Is Kim in the CS building?

Answer with exactly one word: True or False."""

datalog_prompt = """Here is a Datalog program:

in(kim, r123).
part_of(r123, cs_building).
in(X, Y) :- part_of(Z, Y), in(X, Z).

Is the query in(kim,cs_building) true or false for this program?

Answer with exactly one word: True or False."""

print(datalog_prompt)
print("LLM, from English:      ", call_llm(english_prompt))
print("LLM, from the program:  ", call_llm(datalog_prompt))
print("clingo, from the program:", answer(kim_book, "in(kim,cs_building)")[0])

In [ ]:
# The LLM on its own, on the Poole & Mackworth example (gold answer: True).

english_prompt = """Answer the question using only the text. Treat anything not stated as false.

Text: Kim is in room r123. Room r123 is part of the CS building.
Anyone who is in a place is also in anything that place is part of.
Question: Is Kim in the CS building?

Answer with exactly one word: True or False."""

datalog_prompt = """Here is a Datalog program:

in(kim, r123).
part_of(r123, cs_building).
in(X, Y) :- part_of(X, Y), in(X, Z).

Is the query in(kim,cs_building) true or false for this program?

Answer with exactly one word: True or False."""

print(datalog_prompt)
print("LLM, from English:      ", call_llm(english_prompt))
print("LLM, from the program:  ", call_llm(datalog_prompt))
print("clingo, from the program:", answer(kim_book, "in(kim,cs_building)")[0])

**Try it: does the LLM read the program as written?**

In the cell above, change the rule in `datalog_prompt` to the version with one wrong variable:

    in(X, Y) :- part_of(X, Y), in(X, Z).

and change `kim_book` to `kim_typo` in the last line, so clingo checks the same program. Then rerun the cell.

1. What does clingo answer now? What does the LLM answer?
2. If they disagree, which one is right *for this program*? Why might the LLM give the other answer?
3. What does this suggest about using an LLM, rather than a solver, to check whether a program is correct?

### 2.3 A deeper Poole & Mackworth example

The same rule, but now with a six-level chain (room → floor → wing → building → campus → university), the facts in shuffled order, and two people in different wings. The LLM is given the Datalog program and asked about five queries.

In [ ]:
# A deeper version of the Poole & Mackworth example.
campus = """
part_of(wing_b, cs_building).
in(lee, lab7).
part_of(north_campus, university).
part_of(floor3, wing_a).
part_of(r123, floor2).
part_of(annex, north_campus).
part_of(cs_building, north_campus).
in(kim, r123).
part_of(wing_a, cs_building).
part_of(lab7, floor3).
part_of(floor2, wing_b).

in(X, Y) :- part_of(Z, Y), in(X, Z).
"""

queries = ["in(kim,university)",   # True:  6 steps up the chain
           "in(lee,university)",   # True
           "in(lee,wing_b)",       # False: Lee is in wing_a, not wing_b
           "in(kim,wing_a)",       # False: Kim is in wing_b
           "in(kim,annex)"]        # False: the annex is part of the campus, not the other way round

for q in queries:
    llm_ans = call_llm(f"""Here is a Datalog program:
{campus}
Is the query {q} true or false for this program?

Answer with exactly one word: True or False.""")
    print(f"{q:22}  LLM: {llm_ans:6}  clingo: {answer(campus, q)[0]}")

**Discussion.** The LLM answered `in(lee,university)` as **False**, but clingo says **True**.

1. Write out the chain of facts and rule applications that shows Lee is in the university. How many steps is it?
2. The LLM got `in(kim,university)` right, which is a chain of the same length. What might make Lee's chain harder for it to follow?
3. The LLM answered False four times out of five. Can we tell whether its correct False answers came from reasoning, or from a tendency to say False? How could we test this?
4. clingo got every query right and can show which facts it derived. What would we need to trust the LLM's answers the same way?

In [ ]:
CAMPUS_TEXT = (
    "Wing B is part of the CS building. Lee is in lab 7. North campus is part of the university. "
    "Floor 3 is part of wing A. Room r123 is part of floor 2. The annex is part of north campus. "
    "The CS building is part of north campus. Kim is in room r123. Wing A is part of the CS building. "
    "Lab 7 is part of floor 3. Floor 2 is part of wing B. "
    "Anyone who is in a place is also in anything that place is part of."
)

campus_questions = [   # (question in English, matching Datalog query)
    ("Is Kim in the university?", "in(kim,university)"),   # True:  6 steps
    ("Is Lee in the university?", "in(lee,university)"),   # True
    ("Is Lee in wing B?",         "in(lee,wing_b)"),       # False: Lee is in wing A
    ("Is Kim in wing A?",         "in(kim,wing_a)"),       # False: Kim is in wing B
    ("Is Kim in the annex?",      "in(kim,annex)"),        # False
]

for question, q in campus_questions:
    llm_ans = call_llm(f"""Answer the question using only the text. Treat anything not stated as false.

Text: {CAMPUS_TEXT}
Question: {question}

Answer with exactly one word: True or False.""")
    print(f"{question:28}  LLM: {llm_ans:6}  clingo: {answer(campus, q)[0]}")

## Note that Gemma does better with the English than the rules.

In [ ]:
# Same structure as the campus example; every place renamed with a made-up word.
NONSENSE_TEXT = (
    "The quell is part of the vasp. Lee is in the frim. The drell is part of the mobb. "
    "The plonk is part of the grisk. The blick is part of the zorp. The wib is part of the drell. "
    "The vasp is part of the drell. Kim is in the blick. The grisk is part of the vasp. "
    "The frim is part of the plonk. The zorp is part of the quell. "
    "Anyone who is in a place is also in anything that place is part of."
)

nonsense_program = """
part_of(quell, vasp).   in(lee, frim).          part_of(drell, mobb).
part_of(plonk, grisk).  part_of(blick, zorp).   part_of(wib, drell).
part_of(vasp, drell).   in(kim, blick).         part_of(grisk, vasp).
part_of(frim, plonk).   part_of(zorp, quell).
in(X, Y) :- part_of(Z, Y), in(X, Z).
"""

nonsense_questions = [   # (question, query)          campus equivalent
    ("Is Kim in the mobb?",  "in(kim,mobb)"),     # Kim in the university  (True, 6 steps)
    ("Is Lee in the mobb?",  "in(lee,mobb)"),     # Lee in the university  (True)
    ("Is Lee in the quell?", "in(lee,quell)"),    # Lee in wing B          (False)
    ("Is Kim in the grisk?", "in(kim,grisk)"),    # Kim in wing A          (False)
    ("Is Kim in the wib?",   "in(kim,wib)"),      # Kim in the annex       (False)
]

for question, q in nonsense_questions:
    llm_ans = call_llm(f"""Answer the question using only the text. Treat anything not stated as false.

Text: {NONSENSE_TEXT}
Question: {question}

Answer with exactly one word: True or False.""")
    print(f"{question:22}  LLM: {llm_ans:6}  clingo: {answer(nonsense_program, q)[0]}")

### With made up words (and no common-sense reasoning) Gemma defaults to false in all. 

### Same problem, three forms: what is the LLM really doing?

We gave Gemma the deeper Poole & Mackworth problem three ways. The facts, the order of the sentences, and the rule were identical each time; only the wording changed.

| Version | The LLM was given… | LLM correct | Missed |
|---|---|---|---|
| Datalog program | `in(kim, r123).`, `part_of(r123, floor2).`, … | 4 / 5 | Lee in the university |
| English, real places | "Kim is in room r123. Room r123 is part of floor 2. …" | **5 / 5** | none |
| English, made-up words | "Kim is in the blick. The blick is part of the zorp. …" | 3 / 5 (answered False every time) | both True questions |
| clingo (any version, as a program) | the Datalog program | **5 / 5** | none |

**What this shows**

- **With familiar places, the LLM probably wasn't following the six-step chain.** It used what it already knows: someone in a room is in the building, and so in the university. That works only because the rule happens to match common sense.
- **With made-up words, that shortcut is gone, and the LLM fails.** It answered False to every question, so even its three "correct" answers are likely a default guess, not reasoning.
- **clingo gets every version right.** To a solver, `blick` and `r123` are just symbols; the result depends only on the facts and rules.

**Why this matters for the hybrid.** Real problems often don't match common sense: company org charts, legal definitions, game rules, a new course's prerequisites. That is exactly where the LLM alone breaks down. The hybrid gives each part the job it's good at: the **LLM translates** (language and common sense), and the **solver reasons** (exact rules, however unfamiliar or long). This is also why benchmarks like ProntoQA use made-up words: to measure reasoning separately from world knowledge.

**Questions**

1. The LLM got all five right with real place names. Does that mean it can reason through six steps? What other explanation is there?
2. In the made-up version, "is in" and "part of" are still ordinary English. How could you remove even that help? What would you expect to happen?
3. Write one question about a real situation, such as your own course or workplace, where the rules go *against* common sense. Would you trust the LLM alone to answer it?


### 2.4 Deeper problems (D1–D7)

Like the made-up campus example, each of these problems is built so that **common sense can't supply the answer**: it has to come from the rules in the text. They are also longer: the ten basic problems need only one or two reasoning steps, but these need **5 to 7 steps** each. That's the setting where Logic-LM showed the biggest advantage for a solver.

They are designed so that **translation stays easy while the reasoning gets long**:

- **One sentence, one fact or rule.** Each sentence translates directly into a single line of Datalog.
- **Shuffled order.** The sentences don't appear in the order you need them, so the reasoner has to find the chain itself.
- **No help from world knowledge**, in three different ways:

| Problems | Style | How common sense is taken away | What makes it hard |
|---|---|---|---|
| D1, D2 | Category chain, like ProntoQA | **Made-up words**: "Every wumpus is a yumpus." | A 7-step chain. In D2, an exception blocks the chain at step 3. |
| D3, D4 | One-way flights with a loop | **Real cities, invented routes**: knowing geography doesn't help. | D3 needs 7 flights. D4 is False, though the loop makes it look reachable. |
| D5–D7 | Rules with "and" and "not", like ProofWriter | **Real words, arbitrary rules**: "If someone is big and young then they are red." | A 6-step chain. In D6, the chain is blocked only at the very last step. |

There are 4 True and 3 False answers, so a model that always gives the same answer scores at most 57%.


In [ ]:

_ONTOLOGY = ("Every numpus is a tumpus. Every wumpus is a yumpus. Every dumpus is a rompus. "
             "Every zumpus is a dumpus. Every yumpus is a zumpus. Every rompus is a numpus unless it is shiny. "
             "Every vumpus is shiny. Every tumpus is bright. Max is a wumpus. Sally is a vumpus. Sally is a zumpus.")
_ONTOLOGY_GOLD = ("yumpus(X) :- wumpus(X).\nzumpus(X) :- yumpus(X).\ndumpus(X) :- zumpus(X).\n"
                  "rompus(X) :- dumpus(X).\nnumpus(X) :- rompus(X), not shiny(X).\ntumpus(X) :- numpus(X).\n"
                  "bright(X) :- tumpus(X).\nshiny(X) :- vumpus(X).\nwumpus(max).\nvumpus(sally).\nzumpus(sally).")
 
_FLIGHTS = ("Direct flights run one way only. There are direct flights from Ulm to Graz, from Pisa to Oslo, "
            "from Graz to Brno, from Lyon to Pisa, from Brno to Kiel, from Oslo to Riga, from Kiel to Lyon, "
            "and from Riga to Graz. A traveler can reach a city if there is a direct flight to it, or a direct "
            "flight to a city from which it can be reached.")
_FLIGHTS_GOLD = ("flight(ulm,graz).\nflight(pisa,oslo).\nflight(graz,brno).\nflight(lyon,pisa).\n"
                 "flight(brno,kiel).\nflight(oslo,riga).\nflight(kiel,lyon).\nflight(riga,graz).\n"
                 "reach(X,Y) :- flight(X,Y).\nreach(X,Z) :- flight(X,Y), reach(Y,Z).")
 
_RULES = ("If someone is big and young then they are red. All young people are quiet. "
          "If someone is rough and not blue then they are nice. If someone is green then they are big. "
          "If someone is kind and quiet then they are green. If someone is red then they are rough. "
          "Erin is kind. Erin is young. Fiona is young. Fiona is blue. Fiona is kind.")
_RULES_GOLD = ("red(X) :- big(X), young(X).\nquiet(X) :- young(X).\nnice(X) :- rough(X), not blue(X).\n"
               "big(X) :- green(X).\ngreen(X) :- kind(X), quiet(X).\nrough(X) :- red(X).\n"
               "kind(erin).\nyoung(erin).\nyoung(fiona).\nblue(fiona).\nkind(fiona).")
 
DEEP_PROBLEMS = [
 {"id": "D1", "text": _ONTOLOGY, "question": "Is Max bright?",
  "gold": _ONTOLOGY_GOLD, "gold_query": "bright(max)", "label": "True"},      # 7 steps
 {"id": "D2", "text": _ONTOLOGY, "question": "Is Sally bright?",
  "gold": _ONTOLOGY_GOLD, "gold_query": "bright(sally)", "label": "False"},   # blocked at step 3 (Sally is shiny)
 {"id": "D3", "text": _FLIGHTS, "question": "Can a traveler starting in Ulm reach Riga?",
  "gold": _FLIGHTS_GOLD, "gold_query": "reach(ulm,riga)", "label": "True"},   # 7 flights
 {"id": "D4", "text": _FLIGHTS, "question": "Can a traveler starting in Brno reach Ulm?",
  "gold": _FLIGHTS_GOLD, "gold_query": "reach(brno,ulm)", "label": "False"},  # no flight into Ulm
 {"id": "D5", "text": _RULES, "question": "Is Erin nice?",
  "gold": _RULES_GOLD, "gold_query": "nice(erin)", "label": "True"},          # 6 steps
 {"id": "D6", "text": _RULES, "question": "Is Fiona nice?",
  "gold": _RULES_GOLD, "gold_query": "nice(fiona)", "label": "False"},        # blocked only at the last step
 {"id": "D7", "text": _RULES, "question": "Is Fiona rough?",
  "gold": _RULES_GOLD, "gold_query": "rough(fiona)", "label": "True"},        # 5 steps
]
 
for p in DEEP_PROBLEMS:
    got, _ = answer(p["gold"], p["gold_query"])
    print(p["id"], p["label"].ljust(5), "clingo:", got, "" if got == p["label"] else "  <-- MISMATCH")
 

In [ ]:
llm_deep = pd.DataFrame([{"id": p["id"], "question": p["question"], "gold": p["label"], "llm": ask_direct(p)}
                         for p in DEEP_PROBLEMS])
llm_deep["correct"] = llm_deep.llm == llm_deep.gold
display(llm_deep)
print(f"LLM-only accuracy on the deeper problems: {llm_deep.correct.mean():.0%}")
print("How often the LLM answered each way:", llm_deep.llm.value_counts().to_dict())

**Look at more than the accuracy.** Count how often the LLM answered True and how often False. If it gives the same answer to almost every problem, its "correct" answers may come from a default guess rather than reasoning. Use the step-by-step cell in 2.1 with `PID = "D1"` or `"D5"` to see where its reasoning breaks down.

In [ ]:
show_problems(DEEP_PROBLEMS)

In [ ]:
show_problem('D3')

## Part 3: The LLM as translator

Part 2 showed where the LLM is strong and where it is weak:

- **Strong at understanding English.** It reads ordinary sentences well and brings in common sense. With real place names, it answered all five campus questions correctly.
- **Weak as a reasoner.** When the answer has to come from the rules alone (made-up words, invented routes, arbitrary rules, long chains), it falls back on a default guess. It also had trouble applying the closed-world assumption in P1.

So we change the LLM's job. Instead of asking it for the **answer**, we ask it to **translate** the problem into Datalog: one sentence becomes one fact or rule. That plays to its strength, understanding English. clingo then does the reasoning exactly, however long or unfamiliar the chain.

The prompt below spells out the Datalog syntax and asks for a fixed format that we can parse.

In [ ]:
TRANSLATE_PROMPT = """Translate the text and question below into a Datalog program for the clingo system.

Syntax rules:
- Facts: parent(ann,bob).   Rules: ancestor(X,Y) :- parent(X,Y).
- Constants and predicate names are lowercase_with_underscores. Variables start with an uppercase letter.
- Every fact and rule ends with a period. Use "not" for negation and != for "different".
- Treat anything not stated as false (closed-world assumption).
- Use the same predicate name and number of arguments every time.

Output exactly this format and nothing else:
PROGRAM:
<facts and rules, one per line>
QUERY:
<one ground atom that is true exactly when the answer is yes>

Text: {text}
Question: {question}
"""

def parse_translation(reply):
    m = re.search(r"PROGRAM:\s*(.*?)\s*QUERY:\s*(.*)", reply, re.S)
    if not m:
        raise ValueError("Reply must contain 'PROGRAM:' and 'QUERY:' sections.")
    program = m.group(1).strip().strip("`")
    query = m.group(2).strip().strip("`").strip().rstrip(".")
    if not re.fullmatch(r"[a-z]\w*(\([^()]*\))?", query.replace(" ", "")):
        raise ValueError(f"QUERY must be a single ground atom like ancestor(ann,dan); got {query!r}")
    return program, query.replace(" ", "")

def translate(problem, attempt=1, extra=""):
    prompt = TRANSLATE_PROMPT.format(text=problem["text"], question=problem["question"]) + extra
    return call_llm(prompt, demo_key=(problem["id"], "translate", attempt))

print(translate(PROBLEMS[0]))

### Compare translations

Write your own translation of a problem and compare it with the LLM's and the gold version. Different predicate names are fine; what matters is the logic. Common mistakes:

- **Forgetting an exception** ("unless", "except"): a missing `not`.
- **"Every" vs. "some"**: "must pass *every* prerequisite" needs a helper rule with `not`, as in P7.
- **Direction of a relation**: `parent(ann,bob)` vs. `parent(bob,ann)`.
- **Forgetting `X != Y`** when "different" matters.

In [ ]:
PID = "P7"
my_translation = {
    "program": """requires(cs201,cs101).
requires(cs301,cs201).
requires(cs301,math101).
passed(sam,cs101).
passed(sam,cs201).
student(sam).
course(C) :- requires(C,_).
missing(S,C) :- student(S), requires(C,P), not passed(S,P).
can_take(S,C) :- student(S), course(C), not missing(S,C).""",
    "query": "can_take(sam,cs301)",
}
p = next(q for q in PROBLEMS if q["id"] == PID)
llm_prog, llm_q = parse_translation(translate(p))
for name, (prog, q) in {"gold": (p["gold"], p["gold_query"]),
                        "mine": (my_translation["program"], my_translation["query"]),
                        "LLM": (llm_prog, llm_q)}.items():
    print(f"--- {name}: {answer(prog, q)[0]}\n{prog}\n?- {q}\n")

## Part 4: The hybrid system

Now we put the two halves together: the LLM translates the problem into Datalog (Part 3), and clingo does the reasoning. `solve_once` runs one translation through clingo; Part 5 adds the self-repair loop.

In [ ]:
def solve_once(problem):
    program, query = parse_translation(translate(problem))
    result, msg = answer(program, query)
    return result, program, query, msg

for p in PROBLEMS[:3]:
    result, prog, q, msg = solve_once(p)
    print(f"{p['id']}: clingo says {result:6}  (gold: {p['label']})")

## Part 5: Self-repair

When clingo reports an error, or warns that a predicate is never defined, we send the message back and ask the LLM to fix its program. This is the same idea as Logic-LM's self-refiner.

It **can't** catch a program that runs fine but means the wrong thing.

In [ ]:
def solve_with_repair(problem, max_attempts=3, verbose=True):
    extra = ""
    for attempt in range(1, max_attempts + 1):
        reply = translate(problem, attempt, extra)
        try:
            program, query = parse_translation(reply)
            result, msg = answer(program, query)
            pred = query.split("(")[0]
            undefined = msg and re.search(rf"atom does not occur in any rule head:\s*\n?\s*{pred}\b", msg)
            if result == "Error" or undefined:
                raise ValueError(msg)
            return {"answer": result, "attempts": attempt, "program": program, "query": query}
        except ValueError as err:
            if verbose:
                print(f"{problem['id']} attempt {attempt} failed:\n  {str(err).strip()}\n")
            extra = (f"\n\nYour previous answer was:\n{reply}\n\nclingo reported:\n{str(err).strip()}\n"
                     "Fix the problem and output the full corrected answer in the same format.")
    return {"answer": "Failed", "attempts": max_attempts, "program": None, "query": None}

print(solve_with_repair(PROBLEMS[3]))   # P4

In [ ]:
t = time.time()
#print(translate(PROBLEMS[0]))
#print(f"{time.time() - t:.1f} seconds")
for p in PROBLEMS[:3]:
    print(p["id"], solve_with_repair(p), "gold:", p["label"])
print(f"{time.time() - t:.1f} seconds")

In [ ]:
!nvidia-smi

### Seeing self-repair in action

Gemma rarely makes the kind of mistake that triggers repair on the basic problems, so here we **give it a broken translation on purpose** and watch what happens. The cells below run one round of repair on problem P4 ("Is Ann an ancestor of Dan?"):

1. a broken translation,
2. clingo's error or warning,
3. the repair prompt sent back to the LLM,
4. the LLM's corrected translation,
5. clingo's answer on the correction.

We try two common mistakes: the Prolog arrow `<-` instead of `:-`, and a rule that defines `anc` while the query asks about `ancestor`.

In [ ]:
def demo_repair(problem, broken_reply, show_prompt=True):
    """Show one round of self-repair: broken program -> clingo's complaint -> repair prompt -> LLM's fix."""
    print("=== 1. The (broken) translation ===")
    print(broken_reply)
    prog, q = parse_translation(broken_reply)
    result, msg = answer(prog, q)
    print(f"\n=== 2. clingo's verdict: {result} ===")
    print(msg)

    # Same feedback that solve_with_repair sends: the original prompt + previous answer + clingo's message
    extra = (f"\n\nYour previous answer was:\n{broken_reply}\n\nclingo reported:\n{msg}\n"
             "Fix the problem and output the full corrected answer in the same format.")
    if show_prompt:
        repair_prompt = TRANSLATE_PROMPT.format(text=problem["text"], question=problem["question"]) + extra
        print("\n=== 3. The repair prompt sent to the LLM ===")
        print(repair_prompt)

    fixed = translate(problem, attempt=2, extra=extra)
    print("\n=== 4. The LLM's repaired translation ===")
    print(fixed)
    prog2, q2 = parse_translation(fixed)
    print(f"\n=== 5. clingo on the repair: {answer(prog2, q2)[0]}   (gold: {problem['label']}) ===")

In [ ]:
p4 = next(q for q in PROBLEMS if q["id"] == "P4")    # Is Ann an ancestor of Dan?  (True)

# Error 1: Prolog-style arrow "<-" instead of ":-"
demo_repair(p4, """PROGRAM:
parent(ann,bob).
parent(bob,cat).
parent(cat,dan).
ancestor(X,Y) <- parent(X,Y).
ancestor(X,Z) <- ancestor(X,Y), parent(Y,Z).
QUERY:
ancestor(ann,dan)""")

print("\n" + "#" * 60 + "\n")

# Error 2: rules define "anc", but the query asks about "ancestor"
demo_repair(p4, """PROGRAM:
parent(ann,bob).
parent(bob,cat).
parent(cat,dan).
anc(X,Y) :- parent(X,Y).
anc(X,Z) :- anc(X,Y), parent(Y,Z).
QUERY:
ancestor(ann,dan)""")

**What to notice**

- **Error 1:** clingo reports *unsafe variables*. It reads `<-` as "less than" followed by a minus sign, so the message doesn't mention the arrow at all. The LLM still has to work out what's actually wrong. Real error messages are often like this.
- **Error 2:** the program runs, and clingo answers **False**. Without the warning, this wrong answer would go straight through. It's caught only because `answer()` adds the query as a rule, so clingo warns that `ancestor` "does not occur in any rule head."

**What repair can't fix**

Repair only triggers on an error or warning. A program that runs cleanly but means the wrong thing goes straight through. Examples:

- `in(X,Y) :- part_of(X,Y), in(X,Z).` (the Poole & Mackworth example with one wrong variable)
- `flies(X) :- bird(X).` (forgetting "unless it's a penguin")

**Question:** how could the pipeline catch errors like these? (Ideas: generate several translations and compare their answers; ask the LLM to translate the program back into English; test the program on a case where you already know the answer.)

### -------------------

### Repair on a hard problem: P7

The demonstration above used easy mistakes. Here is repair on a real one. P7 ("Can Sam take CS301?") needs "passed **every** required course", which Datalog can only express with a helper predicate and double negation (see the gold program with `show_problem("P7")`).

`verbose=True` prints clingo's message after each failed attempt. Watch how each repair fixes the reported error, and whether the LLM ever finds the right structure.

In [ ]:
r = solve_with_repair(PROBLEMS[6], verbose=True)
print(r["program"], "\n?-", r["query"])

In [ ]:
show_problem("P7")



## Part 6: Evaluation, LLM-only vs. hybrid

In [ ]:
# ask_direct was defined in Part 2.1.
rows = []
for p in PROBLEMS:
    h = solve_with_repair(p, verbose=False)
    rows.append({"id": p["id"], "gold": p["label"], "llm_only": ask_direct(p), "hybrid": h["answer"], "attempts": h["attempts"]})
results = pd.DataFrame(rows)
results["llm_only_ok"] = results.llm_only == results.gold
results["hybrid_ok"] = results.hybrid == results.gold

display(results)
print(f"LLM-only accuracy: {results.llm_only_ok.mean():.0%}    Hybrid accuracy: {results.hybrid_ok.mean():.0%}")

In [ ]:
# Error analysis: show the hybrid's wrong translations next to the gold ones.
for p in PROBLEMS:
    r = solve_with_repair(p, verbose=False)
    if r["answer"] != p["label"]:
        print(f"=== {p['id']}: hybrid said {r['answer']}, gold {p['label']}\n{p['text']} {p['question']}")
        print("LLM program:\n" + str(r["program"]) + f"\n?- {r['query']}")
        print("Gold program:\n" + p["gold"] + f"\n?- {p['gold_query']}\n")

### Evaluation on the deeper problems

Same comparison, LLM-only vs. hybrid, on D1–D7. Does the gap change when the reasoning chains get longer?

In [ ]:
import time
rows = []
start = time.time()
for p in DEEP_PROBLEMS:
    h = solve_with_repair(p, verbose=False)          # hybrid: LLM translates, clingo reasons
    rows.append({"id": p["id"], "question": p["question"], "gold": p["label"],
                 "llm_only": ask_direct(p),          # LLM answers on its own
                 "hybrid": h["answer"], "attempts": h["attempts"]})
    print(f"{p['id']} done ({time.time() - start:.0f} s so far)")

deep = pd.DataFrame(rows)
deep["llm_only_ok"] = deep.llm_only == deep.gold
deep["hybrid_ok"] = deep.hybrid == deep.gold
display(deep)
print(f"LLM-only accuracy: {deep.llm_only_ok.mean():.0%}    Hybrid accuracy: {deep.hybrid_ok.mean():.0%}")

In [ ]:
r = solve_with_repair(DEEP_PROBLEMS[0], verbose=True)   # D1
print(r["program"], "\n?-", r["query"])

r = solve_with_repair(DEEP_PROBLEMS[2], verbose=True)   # D3: prints clingo's error for each attempt

### 
Summary of results

Here is what we found running Gemma 4 E4B on both problem sets. Your numbers may differ a little from run to run.

| Problem set | Reasoning steps | LLM alone | Hybrid (LLM + clingo) |
|---|---|---|---|
| Basic (P1–P10) | 1–2 | **90%** | 80% |
| Deep (D1–D7) | 5–7 | 43% | **71%** |

**On short problems, the LLM alone does fine.** One or two reasoning steps are easy for the model to do "in its head." The hybrid's two misses here were both **translation errors**, not reasoning errors:

- **P7:** "passed **every** required course" needs double negation in Datalog. The first translation had an error. The repair loop produced a program that *runs*, but it likely reads "every" as "some", so clingo answered True instead of False. Repair fixes errors that clingo can detect, not errors of meaning.
- **P8:** "Is Rex (a dog) a mammal?" The text never says dogs are mammals, so the answer is False under the closed-world assumption. The model added that fact from its own knowledge.

**On long problems, the solver pays off.** The hybrid solved 5 of the 7, including 5- and 6-step chains. The LLM alone answered **False to all seven**. Its 43% comes only from the three problems whose answer happens to be False. That isn't reasoning; it's a default guess. Accuracy alone can hide this, so always check *which* answers a model gets right.

**The hybrid's misses on the deep set were again in translation:**

- **D1:** categories became constants (`is_a(numpus, tumpus)`) instead of rules (`tumpus(X) :- numpus(X).`), so clingo couldn't follow the chain.
- **D3:** no usable program after three attempts. The first reply was cut off before it reached the `PROGRAM:` section.

**The big picture.** This matches the Logic-LM paper in miniature: once a problem is correctly translated, the solver's reasoning is always right, however long the chain. The weak link moves to **translation**. So the most promising improvements are better translations: worked examples in the prompt, a wider check for undefined predicates, or a majority vote over several translations.

## Part 7: Datalog vs. first-order logic

| | First-order logic + prover | Datalog + clingo |
|---|---|---|
| Unknown facts | **Unknown** (open world) | **False** (closed world) |
| "Birds fly unless penguins" | hard to state; Tweety is Unknown | easy with `not`; Tweety flies |
| Disjunction ("A or B") | yes | no (in plain Datalog) |
| "There exists someone…" | yes | only by naming them |
| Recursion (ancestor) | awkward to prove | natural and fast |
| Always terminates | no | yes |

**Exercise:** try to translate "Every student takes logic or AI. Sam is a student. Sam doesn't take logic. Does Sam take AI?" into Datalog. What goes wrong, and why? Not possible.